# Preparação dos dados anuais do ICEWS

A [EDA](EDA.ipynb) começa com um ano selecionado em `anos`. Esse recorte ajuda a entender o formato das colunas e conferir como datas, países, intensidades e valores ausentes aparecem nos arquivos.

Na EDA, a tabela mensal usa os nomes dos atores. Para a Parte 2, a unidade do grafo passa a ser o país associado ao ator. Por isso, agrupamos os eventos pelo país de origem e pelo país de destino.

Aqui aplicamos o mesmo procedimento de preparação a cada ano disponível. Conservar o ano permite comparar os períodos e conferir a qualidade dos dados antes de juntar os pares de países em um grafo único. A leitura é feita em blocos. Depois de cada ano, guardamos os resumos e liberamos as linhas brutas.

O procedimento é o mesmo de `resumir_anos.py`. O notebook reaproveita as funções desse script e mostra cada etapa da preparação.

## 1. Escolher os arquivos e os anos

Deixe `anos = None` para processar todos os arquivos anuais encontrados em `raw/`. Para conferir só alguns anos, use uma lista como `[1995, 2017]`. Esse teste grava em `processado_recorte/` para preservar os resumos completos usados na Parte 2.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

pasta_dataset = Path.cwd().resolve()

if not (pasta_dataset / "resumir_anos.py").is_file():
    pasta_dataset = pasta_dataset / "data" / "dataset_parte2"

if not (pasta_dataset / "resumir_anos.py").is_file():
    raise FileNotFoundError("Abra o notebook a partir da raiz do projeto ou de data/dataset_parte2.")

if str(pasta_dataset) not in sys.path:
    sys.path.insert(0, str(pasta_dataset))

from resumir_anos import encontrar_arquivos, processar_ano, verificar_colunas

anos = None
tamanho_parte = 200_000
ignorar_diferencas = False
pasta_raw = pasta_dataset / "raw"

if anos is None:
    pasta_saida = pasta_dataset / "processado"
else:
    pasta_saida = pasta_dataset / "processado_recorte"

In [2]:
if not pasta_raw.is_dir():
    raise FileNotFoundError(f"Pasta de dados brutos não encontrada: {pasta_raw}")

arquivos_encontrados = encontrar_arquivos(pasta_raw)
arquivos_por_ano = {}

if anos is None:
    arquivos_por_ano = arquivos_encontrados
else:
    faltam = []
    for ano in sorted(set(anos)):
        if ano in arquivos_encontrados:
            arquivos_por_ano[ano] = arquivos_encontrados[ano]
        else:
            faltam.append(ano)

    if faltam:
        raise FileNotFoundError(f"Anos sem arquivo em {pasta_raw}: {faltam}")

if not arquivos_por_ano:
    raise FileNotFoundError(f"Nenhum arquivo anual .tab encontrado em {pasta_raw}")

print(f"Anos selecionados: {len(arquivos_por_ano)}")
print(list(arquivos_por_ano))

Anos selecionados: 28
[1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]


## 2. Conferir as colunas

Comparamos os cabeçalhos com o primeiro ano selecionado. Data, país de origem, país de destino e intensidade são necessários para calcular os resumos e montar os pares. O arquivo de conferência registra diferenças de nomes ou de ordem das colunas.

In [3]:
verificacao, referencia = verificar_colunas(arquivos_por_ano)
pasta_saida.mkdir(parents=True, exist_ok=True)
caminho_verificacao = pasta_saida / "verificacao_colunas.csv"
verificacao.to_csv(caminho_verificacao, index=False)
display(verificacao)

sem_colunas_necessarias = verificacao[verificacao["faltam_necessarias"] != ""]
if not sem_colunas_necessarias.empty:
    raise ValueError(f"Há arquivos com colunas obrigatórias ausentes. Consulte {caminho_verificacao}.")

diferentes = verificacao[~verificacao["iguais_a_referencia"]]
if not diferentes.empty and not ignorar_diferencas:
    raise ValueError(f"Há cabeçalhos diferentes. Consulte {caminho_verificacao}.")

print(f"Referência: {len(referencia)} colunas do ano {min(arquivos_por_ano)}.")

,ano,arquivo,qtd_colunas,iguais_a_referencia,mesma_ordem,colunas_faltando,colunas_extras,faltam_necessarias
0,1995,eventos_1995.tab,20,True,True,,,
1,1996,eventos_1996.tab,20,True,True,,,
2,1997,eventos_1997.tab,20,True,True,,,
3,1998,eventos_1998.tab,20,True,True,,,
4,1999,eventos_1999.tab,20,True,True,,,
5,2000,eventos_2000.tab,20,True,True,,,
6,2001,eventos_2001.tab,20,True,True,,,
7,2002,eventos_2002.tab,20,True,True,,,
8,2003,eventos_2003.tab,20,True,True,,,
9,2004,eventos_2004.tab,20,True,True,,,


Referência: 20 colunas do ano 1995.


## 3. Tratar e resumir cada ano

Retiramos espaços dos textos. Campos vazios e o marcador `NULL` entram na contagem de ausências. Registramos separadamente quantos `NULL` vieram escritos no arquivo. Também conferimos datas, conversões de intensidade e duplicatas. Nos resumos anuais, as duplicatas são contabilizadas e as linhas são mantidas.

Para a tabela de pares, selecionamos eventos com intensidade numérica e países de origem e destino identificados e distintos. Agrupamos por origem e destino, nessa ordem. Assim, os dois sentidos de uma relação são contados separadamente. Para cada par e ano, guardamos o número de eventos, a soma e a média das intensidades.

In [4]:
resumos = []
nulos = []
paises = []
pares = []
correlacoes = []

for ano, arquivos in arquivos_por_ano.items():
    print(f"Processando {ano} ({len(arquivos)} arquivo(s))...", flush=True)

    resumo, nulos_ano, paises_ano, pares_ano, correlacao_ano = processar_ano(
        ano, arquivos, tamanho_parte
    )

    resumos.append(resumo)
    nulos.append(nulos_ano)
    paises.append(paises_ano)
    pares.append(pares_ano)
    correlacoes.append(correlacao_ano)

    print(f"  {resumo['eventos']} eventos e {resumo['pares_dirigidos']} pares dirigidos.")

Processando 1995 (1 arquivo(s))...
  151310 eventos e 5661 pares dirigidos.
Processando 1996 (1 arquivo(s))...
  235269 eventos e 6803 pares dirigidos.
Processando 1997 (1 arquivo(s))...
  251425 eventos e 7096 pares dirigidos.
Processando 1998 (1 arquivo(s))...
  368318 eventos e 8147 pares dirigidos.
Processando 1999 (1 arquivo(s))...
  545102 eventos e 9146 pares dirigidos.
Processando 2000 (1 arquivo(s))...
  654613 eventos e 9817 pares dirigidos.
Processando 2001 (1 arquivo(s))...
  797924 eventos e 10418 pares dirigidos.
Processando 2002 (1 arquivo(s))...
  825060 eventos e 10800 pares dirigidos.
Processando 2003 (1 arquivo(s))...
  850489 eventos e 11187 pares dirigidos.
Processando 2004 (1 arquivo(s))...
  953820 eventos e 11884 pares dirigidos.
Processando 2005 (1 arquivo(s))...
  1013400 eventos e 12040 pares dirigidos.
Processando 2006 (1 arquivo(s))...
  1118996 eventos e 12311 pares dirigidos.
Processando 2007 (1 arquivo(s))...
  1011161 eventos e 12001 pares dirigidos.
Pr

## 4. Juntar os resumos preservando o ano

As tabelas anuais são concatenadas com a coluna `ano`. Com ela, ainda é possível comparar a intensidade entre anos, localizar ausências e ver quando cada país ou par aparece. A tabela de presença marca se o país foi registrado como origem ou destino naquele ano.

In [5]:
resumo_df = pd.DataFrame(resumos)
nulos_df = pd.concat(nulos, ignore_index=True)
correlacoes_df = pd.concat(correlacoes, ignore_index=True)
paises_df = pd.concat(paises, ignore_index=True)
pares_df = pd.concat(pares, ignore_index=True)

paises_presentes = paises_df.assign(presente=True)
presenca = paises_presentes.pivot_table(
    index="pais",
    columns="ano",
    values="presente",
    aggfunc="any",
    fill_value=False,
)
presenca = presenca.astype(bool)
presenca.columns = presenca.columns.astype(str)
presenca["anos_presente"] = presenca.sum(axis=1)

colunas_resumo = ["ano", "eventos", "paises", "pares_dirigidos", "pares_media_negativa"]
display(resumo_df[colunas_resumo])
display(pares_df.head())

,ano,eventos,paises,pares_dirigidos,pares_media_negativa
0,1995,151310,221,5661,691
1,1996,235269,227,6803,714
2,1997,251425,229,7096,779
3,1998,368318,231,8147,886
4,1999,545102,231,9146,906
5,2000,654613,236,9817,1007
6,2001,797924,239,10418,1102
7,2002,825060,237,10800,920
8,2003,850489,238,11187,988
9,2004,953820,239,11884,1294


,ano,origem,destino,n_eventos,soma_intensidade,media_intensidade
0,1995,Afghanistan,France,1,2.8,2.800000
1,1995,Afghanistan,Germany,3,-3.0,-1.000000
2,1995,Afghanistan,India,3,8.4,2.800000
3,1995,Afghanistan,Iran,38,90.6,2.384211
4,1995,Afghanistan,Malaysia,1,4.0,4.000000


## 5. Salvar os arquivos

| Arquivo | O que guarda |
|---|---|
| `verificacao_colunas.csv` | Conferência dos cabeçalhos, salva na etapa 2 |
| `resumo_anos.csv` | Contagens e estatísticas de cada ano |
| `nulos_por_ano.csv` | Ausências por coluna e ano, incluindo as escritas como `NULL` |
| `correlacao_nulos_por_ano.csv` | Correlação entre indicadores de ausência, por ano |
| `presenca_paises.csv` | Presença de cada país por ano |
| `eventos_por_pais_ano.csv` | Contagens como origem e como destino |
| `pares_paises_por_ano.csv.gz` | Contagens, somas e médias de intensidade por par dirigido e ano |

In [6]:
resumo_df.to_csv(pasta_saida / "resumo_anos.csv", index=False)
nulos_df.to_csv(pasta_saida / "nulos_por_ano.csv", index=False)
correlacoes_df.to_csv(pasta_saida / "correlacao_nulos_por_ano.csv", index=False)
presenca.to_csv(pasta_saida / "presenca_paises.csv")
paises_df.to_csv(pasta_saida / "eventos_por_pais_ano.csv", index=False)
pares_df.to_csv(pasta_saida / "pares_paises_por_ano.csv.gz", index=False)

print(f"Sete arquivos salvos em {pasta_saida}.")

Sete arquivos salvos em C:\Users\caiom\OneDrive\Desktop\projeto grafos\projeto-grafos\data\dataset_parte2\processado.


## Do resumo anual ao grafo da Parte 2

Depois de gerar os resumos completos, `gerar_arestas.py` usa `processado/pares_paises_por_ano.csv.gz` para criar `arestas_paises.csv`. O script padroniza nomes de países e soma, para cada par dirigido, as intensidades e as contagens dos anos. Dividir a soma total pelo número total de eventos dá o peso da aresta. Assim, cada evento tem o mesmo peso nessa média, mesmo quando os anos têm quantidades diferentes de registros.
